In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from option import Option

from pricing_methods.black_scholes_analytic import analytic_bs_pricer
from pricing_methods.binomial import binomial_pricer
from pricing_methods.monte_carlo import mc_pricer
from pricing_methods.finite_difference import fd_pricer
from pricing_methods.pinn import pinn_pricer, build_model, train, make_training_step_am, payoff

Constructing the data from Broadie & Detemple (1996)

In [2]:
K = 100
T = 3.0

In [3]:
# In the form (r, sigma, div_yield)
option_params = [(0.03, 0.2, 0.07), (0.03, 0.4, 0.07), (0.0, 0.3, 0.07), (0.07, 0.3, 0.03)]

In [4]:
params = []
for p in option_params:
    for i in [80, 90, 100, 110, 120]:
        temp = [p[0], p[1], p[2], i, None]
        params.append(temp)

In [5]:
params[0][-1] = 2.580
params[1][-1] = 5.167
params[2][-1] = 9.066
params[3][-1] = 14.443
params[4][-1] = 21.414
params[5][-1] = 11.326
params[6][-1] = 15.722
params[7][-1] = 20.793
params[8][-1] = 26.495
params[9][-1] = 32.781
params[10][-1] = 5.518
params[11][-1] = 8.842
params[12][-1] = 13.142
params[13][-1] = 18.453
params[14][-1] = 24.791
params[15][-1] = 12.145
params[16][-1] = 17.369
params[17][-1] = 23.348
params[18][-1] = 29.964
params[19][-1] = 37.104

In [6]:
bin_errors = []
mc_errors = []
fd_errors = []
pinn_errors = []

for p in params:
    r = p[0]
    sigma = p[1]
    div_yield = p[2]
    S0 = p[3]
    true_price = p[4]

    option = Option(S0, K, T, r, sigma, div_yield, "call", "american")

    bin_price = binomial_pricer(option, n_steps=300)
    mc_price = mc_pricer(option, n_paths=200000, n_steps=100)
    fd_price = fd_pricer(option, n_space=300, n_steps=300)

    bin_error = np.abs(bin_price - true_price)
    mc_error = np.abs(mc_price - true_price)
    fd_error = np.abs(fd_price - true_price)

    bin_errors.append(bin_error)
    mc_errors.append(mc_error)
    fd_errors.append(fd_error)


In [7]:
for i in range(len(params[0:5]) // 5):
    p = params[i * 5]
    r = p[0]
    sigma = p[1]
    div_yield = p[2]
    S0 = p[3]

    option = Option(S0, K, T, r, sigma, div_yield, "call", "american")

    S0 = option.S0
    K = option.K
    T = option.T
    option_type = option.option_type

    model = build_model(option, 64, 4)

    lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=1e-3,
        decay_steps=5000,
        decay_rate=0.5)

    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)

    training_step = make_training_step_am(option, model, optimizer)
    train(training_step, option, 5000, 500, 500, Smin=0.05, Smax=300, epochs=5000, resample_rate=100, verbose=False)

    for j in range(5):
        cur_S0 = params[i*5 + j][3]
        cur_true_price = params[i*5 + j][4]

        price = float(model(tf.constant([[cur_S0, 0.0]], dtype=tf.float32))[0, 0])
        intrinsic = float(payoff(cur_S0, K, option_type))

        pinn_price = max(price, intrinsic)
        pinn_error = np.abs(pinn_price - cur_true_price)
        pinn_errors.append(pinn_error)

In [8]:
# p = params[0]

# r = p[0]
# sigma = p[1]
# div_yield = p[2]
# S0 = p[3]
# true_price = p[4]

# option = Option(S0, K, T, r, sigma, div_yield, "call", "american")


In [9]:
# pinn_price = pinn_pricer(option, epochs=10000, n_collocation=1000, n_initial=100, n_boundary=100, resample_rate=100)
# pinn_error = np.abs(pinn_price - true_price)

In [10]:
# print(pinn_errors)

In [11]:
print(np.mean(bin_errors))
print(np.mean(mc_errors))
print(np.mean(fd_errors))
print(np.mean(pinn_errors))

0.007018202695674503
0.09676617686108438
0.0013450879374984127
0.8777354953765866
